# 2.3 — Model Evaluation & 5-Fold Cross-Validation
**AIKONIC — Evaluation**

This notebook:
- Runs inference on the **test set** (original unaugmented Mendeley images only)
- Computes Accuracy, Precision, Sensitivity, Specificity, F1, AUROC
- Runs **5-Fold Stratified CV** on the train+val pool
- Saves all evaluation artifacts to `evaluation/`
- Writes `reports/evaluation.md`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

import config
from data_loader import DataLoader

os.makedirs(config.EVAL_DIR, exist_ok=True)
print(f'Evaluation output → {config.EVAL_DIR}')

In [ ]:
# ── Load production model ──────────────────────────────────────────────────────
assert os.path.exists(config.CHECKPOINT_PROD), (
    f'Production model not found: {config.CHECKPOINT_PROD}\n'
    'Run 2.2_model_training.ipynb first.'
)
model = tf.keras.models.load_model(config.CHECKPOINT_PROD)
print(f'✓ Model loaded from {config.CHECKPOINT_PROD}')

In [ ]:
# ── Load test data ─────────────────────────────────────────────────────────────
loader = DataLoader()

# Load full test split as arrays (original, unaugmented Mendeley samples only)
X_test, y_test = loader.get_arrays('test')

n_test = len(y_test)
print(f'Test samples: {n_test}  (PD={int((y_test==1).sum())}  LPD={int((y_test==0).sum())})')

In [ ]:
# ── Run inference ──────────────────────────────────────────────────────────────
print('Running inference on test set...')
y_pred_probs   = model.predict(X_test, verbose=1)   # (N, 2)
y_pred         = np.argmax(y_pred_probs, axis=1)    # (N,)
y_pred_pd_prob = y_pred_probs[:, 1]                 # P(PD) for ROC

print(f'Predictions: PD={int((y_pred==1).sum())}  LPD={int((y_pred==0).sum())}')

In [ ]:
# ── Compute metrics ────────────────────────────────────────────────────────────
acc         = accuracy_score(y_test, y_pred)
precision   = precision_score(y_test, y_pred, zero_division=0)
sensitivity = recall_score(y_test, y_pred, zero_division=0)
f1          = f1_score(y_test, y_pred, zero_division=0)
auroc       = roc_auc_score(y_test, y_pred_pd_prob)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print(f'┌─────────────────────────────────┐')
print(f'│  TEST SET METRICS               │')
print(f'├─────────────────────────────────┤')
print(f'│  Accuracy    : {acc:.4f}          │')
print(f'│  Precision   : {precision:.4f}          │')
print(f'│  Sensitivity : {sensitivity:.4f}   {"✓" if sensitivity >= config.SENSITIVITY_THRESHOLD else "⚠ BELOW TARGET"}    │')
print(f'│  Specificity : {specificity:.4f}          │')
print(f'│  F1-Score    : {f1:.4f}          │')
print(f'│  AUROC       : {auroc:.4f}          │')
print(f'└─────────────────────────────────┘')
print(f'  CM: TP={tp}  TN={tn}  FP={fp}  FN={fn}')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['LPD', 'PD']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title(
    f'Confusion Matrix (n={n_test})\n'
    f'Acc={acc:.3f}  Sens={sensitivity:.3f}  Spec={specificity:.3f}',
    fontsize=11
)
plt.tight_layout()
cm_path = os.path.join(config.EVAL_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'✓ Confusion matrix → {cm_path}')

In [ ]:
# ── ROC Curve ──────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_pred_pd_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auroc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Test Set')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
roc_path = os.path.join(config.EVAL_DIR, 'roc_curve.png')
plt.savefig(roc_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'✓ ROC curve → {roc_path}')

In [ ]:
# ── Learning curves from training CSV ─────────────────────────────────────────
if os.path.exists(config.TRAINING_METRICS):
    metrics_df = pd.read_csv(config.TRAINING_METRICS)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (col, val_col, title) in zip(axes, [
        ('loss',     'val_loss',     'Loss'),
        ('accuracy', 'val_accuracy', 'Accuracy'),
    ]):
        if col in metrics_df.columns:
            ax.plot(metrics_df['epoch'] + 1, metrics_df[col],     label='Train', lw=2)
        if val_col in metrics_df.columns:
            ax.plot(metrics_df['epoch'] + 1, metrics_df[val_col], label='Val',   lw=2, linestyle='--')
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle('Learning Curves (Phase A + Phase B)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    lc_path = os.path.join(config.EVAL_DIR, 'learning_curves.png')
    plt.savefig(lc_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✓ Learning curves → {lc_path}')
else:
    print(f'⚠  {config.TRAINING_METRICS} not found — skipping learning curves')

In [ ]:
# ── Classification Report ──────────────────────────────────────────────────────
report_str = classification_report(
    y_test, y_pred, target_names=config.CLASS_NAMES, zero_division=0
)
print(report_str)

report_path = os.path.join(config.EVAL_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f'AIKONIC — Classification Report (Test Set)\n')
    f.write(f'Model: {config.CHECKPOINT_PROD}\n')
    f.write(f'Test samples: {n_test}\n\n')
    f.write(report_str)
print(f'✓ Classification report → {report_path}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  5-FOLD CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print('5-Fold Stratified Cross-Validation on train+val pool...')

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Small

def build_model_cv(learning_rate=config.PHASE_B_LR):
    """Fresh model for each CV fold (full unfreeze)."""
    inputs = tf.keras.Input(shape=(224, 224, 1), name='grayscale_input')
    x = layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1), name='channel_replication')(inputs)
    base = MobileNetV3Small(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = True
    x = base(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(config.DENSE_UNITS, activation=config.ACTIVATION)(x)
    x = layers.Dropout(config.DROPOUT_RATE)(x)
    outputs = layers.Dense(config.NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall')],
    )
    return model

# Combine train + val arrays
X_train, y_train = loader.get_arrays('train')
X_val,   y_val   = loader.get_arrays('val')
X_cv = np.concatenate([X_train, X_val], axis=0)
y_cv = np.concatenate([y_train, y_val], axis=0)
print(f'CV pool: {len(X_cv)} samples | LPD={int((y_cv==0).sum())} PD={int((y_cv==1).sum())}')

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_SEED)
fold_results = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_cv, y_cv), start=1):
    print(f'\nFold {fold}/5 ─────────────────────────────────')
    X_tr, X_vl = X_cv[tr_idx], X_cv[vl_idx]
    y_tr, y_vl = y_cv[tr_idx], y_cv[vl_idx]

    y_tr_cat = tf.keras.utils.to_categorical(y_tr, 2).astype('float32')
    y_vl_cat = tf.keras.utils.to_categorical(y_vl, 2).astype('float32')

    cw_arr  = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
    cw_fold = {int(c): float(w) for c, w in zip(np.unique(y_tr), cw_arr)}

    fold_model = build_model_cv()
    fold_model.fit(
        X_tr, y_tr_cat,
        validation_data=(X_vl, y_vl_cat),
        epochs=30,
        batch_size=config.BATCH_SIZE,
        class_weight=cw_fold,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True, verbose=0
        )],
        verbose=0,
    )

    probs_vl  = fold_model.predict(X_vl, verbose=0)
    y_pred_vl = np.argmax(probs_vl, axis=1)
    fold_acc   = accuracy_score(y_vl, y_pred_vl)
    fold_sens  = recall_score(y_vl, y_pred_vl, zero_division=0)
    fold_auroc = roc_auc_score(y_vl, probs_vl[:, 1]) if len(np.unique(y_vl)) > 1 else float('nan')

    fold_results.append({
        'fold': fold, 'accuracy': fold_acc,
        'sensitivity': fold_sens, 'auroc': fold_auroc,
        'n_train': len(X_tr), 'n_val': len(X_vl),
    })
    print(f'  Fold {fold} | Acc={fold_acc:.4f}  Sens={fold_sens:.4f}  AUROC={fold_auroc:.4f}')
    tf.keras.backend.clear_session()

In [ ]:
# ── Save CV results ────────────────────────────────────────────────────────────
cv_df = pd.DataFrame(fold_results)
mean_row = {'fold': 'mean', 'accuracy': cv_df['accuracy'].mean(),
            'sensitivity': cv_df['sensitivity'].mean(), 'auroc': cv_df['auroc'].mean(),
            'n_train': '', 'n_val': ''}
std_row  = {'fold': 'std',  'accuracy': cv_df['accuracy'].std(),
            'sensitivity': cv_df['sensitivity'].std(), 'auroc': cv_df['auroc'].std(),
            'n_train': '', 'n_val': ''}
cv_df = pd.concat([cv_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

cv_path = os.path.join(config.EVAL_DIR, 'cv5_fold_results.csv')
cv_df.to_csv(cv_path, index=False)
print(f'✓ CV results → {cv_path}')
print(cv_df.to_string(index=False))

In [ ]:
# ── Write evaluation.md ────────────────────────────────────────────────────────
cv_mean_acc   = float(cv_df[cv_df['fold']=='mean']['accuracy'].values[0])
cv_std_acc    = float(cv_df[cv_df['fold']=='std']['accuracy'].values[0])
cv_mean_sens  = float(cv_df[cv_df['fold']=='mean']['sensitivity'].values[0])
cv_std_sens   = float(cv_df[cv_df['fold']=='std']['sensitivity'].values[0])
cv_mean_auroc = float(cv_df[cv_df['fold']=='mean']['auroc'].values[0])
cv_std_auroc  = float(cv_df[cv_df['fold']=='std']['auroc'].values[0])

eval_md = f"""# AIKONIC — Model Evaluation Report

## Test Set Metrics (Original Unaugmented Mendeley — n={n_test})

| Metric      | Value  |
|-------------|--------|
| Accuracy    | {acc:.4f} |
| Precision   | {precision:.4f} |
| Sensitivity | {sensitivity:.4f} |
| Specificity | {specificity:.4f} |
| F1-Score    | {f1:.4f} |
| AUROC       | {auroc:.4f} |

Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}

## 5-Fold Cross-Validation (train+val pool — n={len(X_cv)})

| Metric      | Mean ± Std |
|-------------|------------|
| Accuracy    | {cv_mean_acc:.4f} ± {cv_std_acc:.4f} |
| Sensitivity | {cv_mean_sens:.4f} ± {cv_std_sens:.4f} |
| AUROC       | {cv_mean_auroc:.4f} ± {cv_std_auroc:.4f} |

## Honest Disclosure
- Single-source dataset: All 249 samples from Mendeley (Ramlan et al., 2023)
- Small sample size produces wide confidence intervals
- Sensitivity target {'MET' if sensitivity >= config.SENSITIVITY_THRESHOLD else 'NOT MET'}: {sensitivity:.4f} vs threshold {config.SENSITIVITY_THRESHOLD:.2f}
"""

os.makedirs(config.REPORTS_DIR, exist_ok=True)
eval_md_path = os.path.join(config.REPORTS_DIR, 'evaluation.md')
with open(eval_md_path, 'w') as f:
    f.write(eval_md)
print(f'✓ evaluation.md → {eval_md_path}')

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print('=' * 60)
print('  EVALUATION COMPLETE')
print('=' * 60)
print(f'  Test : Acc={acc:.4f}  Sens={sensitivity:.4f}  AUROC={auroc:.4f}')
print(f'  CV   : Acc={cv_mean_acc:.4f}±{cv_std_acc:.4f}  Sens={cv_mean_sens:.4f}±{cv_std_sens:.4f}')
print(f'  Artifacts → {config.EVAL_DIR}/')
print()